# Aeon CNN for Time Series Classification

This notebook demonstrates how to use the Aeon CNN model for time series classification.

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Add the parent directory to the path to import from src
sys.path.append("..")

# Set random seed for reproducibility
np.random.seed(42)

## 1. Load and Prepare Data

In [ ]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset for time series classification.

    Args:
        file_path: Path to the .npy file containing the dataset

    Returns:
        Tuple containing (X_train, y_train, X_test, y_test)
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    return X_train, y_train, X_test, y_test

In [ ]:
# Specify the path to your dataset
dataset_path = "CMJ.npy"  # Change this to your dataset path

# Load the dataset
X_train, y_train, X_test, y_test = load_npy_dataset(dataset_path)

## 2. Visualize Some Examples

In [ ]:
def plot_time_series_examples(X, y, n_examples=3):
    """Plot a few examples of time series from each class."""
    classes = np.unique(y)
    n_classes = len(classes)

    plt.figure(figsize=(15, n_classes * 3))

    for i, cls in enumerate(classes):
        # Get indices of examples from this class
        idx = np.where(y == cls)[0][:n_examples]

        for j, example_idx in enumerate(idx):
            plt.subplot(n_classes, n_examples, i * n_examples + j + 1)

            if len(X.shape) == 3:  # Multivariate
                for dim in range(X.shape[1]):
                    plt.plot(X[example_idx, :, dim], label=f"Dim {dim}")
                if X.shape[1] > 1:
                    plt.legend(loc="upper right")
            else:  # Univariate
                plt.plot(X[example_idx])

            plt.title(f"Class {cls}")
            plt.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize examples
plot_time_series_examples(X_train, y_train)

## 3. Import and Configure Aeon CNN Model

In [ ]:
from aeon.classification.deep_learning import CNNTimeClassifier

# Alternative: Import our wrapper class
# from src.models.aeon_cnn.model import AeonCNNModel

In [ ]:
# Initialize the CNNClassifier with desired parameters
clf = CNNClassifier(
    n_epochs=500,  # Number of training epochs
    batch_size=16,  # Batch size
    kernel_size=7,  # Size of convolutional kernel
    n_filters=16,  # Number of convolutional filters
    random_state=42,  # For reproducibility
)

## 4. Train the Model

In [ ]:
# Record the start time
start_time = time.time()

# Fit the model
clf.fit(X_train, y_train)

# Calculate training time
fit_time = time.time() - start_time
print(f"Training completed in {fit_time:.2f} seconds")

## 5. Evaluate the Model

In [ ]:
# Measure prediction time for training data
start_pred_train_time = time.time()
y_pred_train = clf.predict(X_train)
pred_train_time = time.time() - start_pred_train_time

# Measure prediction time for test data
start_pred_test_time = time.time()
y_pred_test = clf.predict(X_test)
pred_test_time = time.time() - start_pred_test_time

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Training Prediction Time: {pred_train_time:.4f} seconds")
print(f"Test Prediction Time: {pred_test_time:.4f} seconds")

In [ ]:
# Calculate samples per second
train_samples_per_second = X_train.shape[0] / pred_train_time
test_samples_per_second = X_test.shape[0] / pred_test_time

print(f"Training Samples per Second: {train_samples_per_second:.2f}")
print(f"Test Samples per Second: {test_samples_per_second:.2f}")

In [ ]:
# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred_test))

## 6. Visualize Confusion Matrix

In [ ]:
import seaborn as sns

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred_test)
classes = np.unique(y_test)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## 7. Save Model and Results

In [ ]:
import pickle
import json
from datetime import datetime

# Create results directory if it doesn't exist
os.makedirs("../results", exist_ok=True)

# Save the model
with open(
    f"../results/aeon_cnn_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl", "wb"
) as f:
    pickle.dump(clf, f)

# Compile results
results = {
    "model_name": "AeonCNN",
    "train_accuracy": float(train_accuracy),
    "test_accuracy": float(test_accuracy),
    "fit_time": float(fit_time),
    "pred_train_time": float(pred_train_time),
    "pred_test_time": float(pred_test_time),
    "train_samples_per_second": float(train_samples_per_second),
    "test_samples_per_second": float(test_samples_per_second),
    "dataset_name": os.path.basename(dataset_path).split(".")[0],
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

# Save results as JSON
with open(
    f"../results/aeon_cnn_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json", "w"
) as f:
    json.dump(results, f, indent=4)

## 8. Extended Analysis (Optional)

In [ ]:
# Get model size in bytes
def get_model_size(model):
    """Estimate the size of the model in bytes"""
    import pickle
    import sys

    return sys.getsizeof(pickle.dumps(model))


model_size_bytes = get_model_size(clf)
print(f"Model Size: {model_size_bytes / 1024:.2f} KB")

In [ ]:
# Analyze incorrectly classified instances
incorrect_indices = np.where(y_pred_test != y_test)[0]
print(f"Number of misclassified instances: {len(incorrect_indices)}")

if len(incorrect_indices) > 0:
    # Display a few misclassified instances
    n_examples = min(3, len(incorrect_indices))
    plt.figure(figsize=(15, n_examples * 3))

    for i in range(n_examples):
        idx = incorrect_indices[i]
        plt.subplot(n_examples, 1, i + 1)

        if len(X_test.shape) == 3:  # Multivariate
            for dim in range(X_test.shape[1]):
                plt.plot(X_test[idx, :, dim], label=f"Dim {dim}")
            if X_test.shape[1] > 1:
                plt.legend(loc="upper right")
        else:  # Univariate
            plt.plot(X_test[idx])

        plt.title(f"True: {y_test[idx]}, Predicted: {y_pred_test[idx]}")
        plt.grid(True)

    plt.tight_layout()
    plt.show()

In [1]:
def load_npy_dataset(file_path):
    """
    Load and prepare numpy dataset for time series classification.

    Args:
        file_path: Path to the .npy file containing the dataset

    Returns:
        Tuple containing (X_train, y_train, X_test, y_test)
    """
    data = np.load(file_path, allow_pickle=True).item()

    X_train = data["train"]["X"]
    y_train = np.array([int(x) for x in data["train"]["y"]])
    X_test = data["test"]["X"]
    y_test = np.array([int(x) for x in data["test"]["y"]])

    print(f"Original shapes:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    # Convert from (B,C,T) to (B,T,C) for aeon
    if len(X_train.shape) == 3:
        X_train = np.transpose(X_train, (0, 2, 1))
        X_test = np.transpose(X_test, (0, 2, 1))
        print(f"\nConverted to aeon format (B,T,C):")
        print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

    print(f"Classes: {np.unique(y_train)}")
    return X_train, y_train, X_test, y_test

In [2]:
import numpy as np
import time
from sklearn.metrics import accuracy_score
from aeon.classification.deep_learning import (
    InceptionTimeClassifier,
    DisjointCNNClassifier,
    LITETimeClassifier,
)

In [3]:
import numpy as np
import time
from sklearn.metrics import accuracy_score
from aeon.classification.deep_learning import (
    InceptionTimeClassifier,
    LITETimeClassifier,
)

# Specify the path to your dataset
datasets = [
    "MP8.npy",
    "MP50.npy",
    "CMJ.npy",
    "synth.npy",
]  # Change this to your dataset path


def run_stronger_baselines(datasets):
    """
    Run stronger baselines: InceptionTime, DisjointCNN, LITETime, and LITEMVTime
    """
    results = {}

    # Define models
    models = {
        "InceptionTime": InceptionTimeClassifier(
            n_epochs=10, batch_size=32, random_state=42, verbose=False
        ),
        "LITEMVTime": LITETimeClassifier(
            n_classifiers=5,
            use_litemv=True,  # Multivariate version
            n_epochs=10,
            batch_size=32,
            random_state=42,
            verbose=False,
        ),
        "DisjointCNN": DisjointCNNClassifier(
            n_epochs=10,
            batch_size=32,
            random_state=42,
            verbose=False,
        ),
    }

    for dataset_name in datasets:
        print(f"\n=== Processing {dataset_name} ===")

        # Load data
        X_train, y_train, X_test, y_test = load_npy_dataset(dataset_name)

        print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
        print(f"Classes: {np.unique(y_train)}")

        dataset_results = {}

        for model_name, model in models.items():
            print(f"\nTraining {model_name}...")

            try:
                # Training
                start_time = time.time()
                model.fit(X_train, y_train)
                train_time = time.time() - start_time

                # Get actual epochs trained (if available)
                try:
                    if hasattr(model, "history") and model.history:
                        epochs_trained = len(model.history.history["loss"])
                    else:
                        epochs_trained = 500  # fallback
                except:
                    epochs_trained = 500  # fallback

                # Prediction
                start_time = time.time()
                y_pred = model.predict(X_test)
                pred_time = time.time() - start_time

                # Calculate accuracy
                accuracy = accuracy_score(y_test, y_pred)

                dataset_results[model_name] = {
                    "accuracy": accuracy,
                    "epochs": epochs_trained,
                    "train_time": train_time,
                    "pred_time": pred_time,
                }

                print(
                    f"{model_name} - Accuracy: {accuracy:.3f}, "
                    f"Epochs: {epochs_trained}, "
                    f"Train: {train_time:.2f}s, Pred: {pred_time:.2f}s"
                )

            except Exception as e:
                print(f"Error with {model_name}: {str(e)}")
                dataset_results[model_name] = {
                    "accuracy": 0.0,
                    "epochs": 0,
                    "train_time": 0.0,
                    "pred_time": 0.0,
                    "error": str(e),
                }

        results[dataset_name] = dataset_results

    return results


def print_latex_table(results):
    """
    Print results in LaTeX table format for adding to Table 3
    """
    print("\n=== LaTeX Table Addition ===")
    print("% Add these rows to Table 3: Deep Learning Methods")

    for dataset, models in results.items():
        for model, metrics in models.items():
            if "error" not in metrics:
                print(
                    f"{dataset} & {model} & {metrics['accuracy']:.3f} & "
                    f"{metrics['epochs']} & {metrics['train_time']:.2f} & "
                    f"{metrics['pred_time']:.2f} \\\\"
                )


def run_multiple_seeds(datasets, n_runs=3):
    """
    Run experiments with multiple random seeds for variance reporting
    """
    all_results = []

    for run in range(n_runs):
        print(f"\n{'=' * 50}")
        print(f"RUN {run + 1}/{n_runs} (seed: {42 + run})")
        print(f"{'=' * 50}")

        # Update random seeds for each run
        models = {
            "InceptionTime": InceptionTimeClassifier(
                n_epochs=500, batch_size=32, random_state=42 + run, verbose=False
            ),
            "LITEMVTime": LITETimeClassifier(
                n_classifiers=5,
                use_litemv=True,
                n_epochs=500,
                batch_size=32,
                random_state=42 + run,
                verbose=False,
            ),
        }

        run_results = {}
        for dataset_name in datasets.items():
            print(f"\n--- Processing {dataset_name} ---")

            X_train, X_test, y_train, y_test = load_npy_dataset(dataset_name)
            dataset_results = {}

            for model_name, model in models.items():
                print(f"Training {model_name}...")

                try:
                    start_time = time.time()
                    model.fit(X_train, y_train)
                    train_time = time.time() - start_time

                    start_time = time.time()
                    y_pred = model.predict(X_test)
                    pred_time = time.time() - start_time

                    accuracy = accuracy_score(y_test, y_pred)

                    dataset_results[model_name] = {
                        "accuracy": accuracy,
                        "train_time": train_time,
                        "pred_time": pred_time,
                    }

                    print(f"  {model_name}: {accuracy:.3f}")

                except Exception as e:
                    print(f"  Error with {model_name}: {str(e)}")
                    dataset_results[model_name] = {"accuracy": 0.0, "error": str(e)}

            run_results[dataset_name] = dataset_results

        all_results.append(run_results)

    return all_results


def compute_statistics(all_results):
    """
    Compute mean and std from multiple runs
    """
    import numpy as np

    # Extract unique datasets and models
    datasets = list(all_results[0].keys())
    models = list(all_results[0][datasets[0]].keys())

    stats = {}

    for dataset in datasets:
        stats[dataset] = {}
        for model in models:
            accuracies = []
            for run_result in all_results:
                if "error" not in run_result[dataset][model]:
                    accuracies.append(run_result[dataset][model]["accuracy"])

            if accuracies:
                stats[dataset][model] = {
                    "mean_acc": np.mean(accuracies),
                    "std_acc": np.std(accuracies),
                    "n_runs": len(accuracies),
                }
            else:
                stats[dataset][model] = {"mean_acc": 0.0, "std_acc": 0.0, "n_runs": 0}

    return stats


def print_latex_with_variance(stats):
    """
    Print LaTeX table with mean ± std
    """
    print("\n=== LaTeX Table with Variance ===")
    print("% Format: mean ± std")

    for dataset, models in stats.items():
        for model, metrics in models.items():
            if metrics["n_runs"] > 0:
                print(
                    f"{dataset} & {model} & "
                    f"{metrics['mean_acc']:.3f} ± {metrics['std_acc']:.3f} & "
                    f"500 & XX.XX & X.XX \\\\"
                )


# Example usage
if __name__ == "__main__":
    # Define your dataset paths
    datasets = ["MP8.npy"]  # Change this to your dataset path

    # Option 1: Single run (faster for testing)
    print("=== SINGLE RUN ===")
    results = run_stronger_baselines(datasets)
    print_latex_table(results)

    # Option 2: Multiple runs for variance (more reliable)
    # print("\n=== MULTIPLE RUNS FOR VARIANCE ===")
    # all_results = run_multiple_seeds(datasets, n_runs=3)
    # stats = compute_statistics(all_results)
    # print_latex_with_variance(stats)

    # Save all results
    import json

    with open("stronger_baselines_results.json", "w") as f:
        json.dump(
            {"single_run": results},
            f,
            indent=2,
        )

=== SINGLE RUN ===

=== Processing MP8.npy ===
Original shapes:
X_train shape: (1426, 8, 161)
X_test shape: (595, 8, 161)
y_train shape: (1426,)
y_test shape: (595,)

Converted to aeon format (B,T,C):
X_train: (1426, 161, 8), X_test: (595, 161, 8)
Classes: [0 1 2 3]
Train shape: (1426, 161, 8), Test shape: (595, 161, 8)
Classes: [0 1 2 3]

Training InceptionTime...
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
InceptionTime - Accuracy: 0.792, Epochs: 500, Train: 57.89s, Pred: 1.83s

Training LITEMVTime...
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
LITEMVTime - Accuracy: 0.755, Epochs: 500, Train: 59.09s, Pred: 1.58s

Training DisjointCNN...
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
